In [ ]:
import os
import shutil
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from google.colab import drive
drive.mount('/content/drive')

IMG_SIZE    = 320
CLASS_NAMES = ["grade0", "grade1", "grade2", "grade3"]
NA          = 3
STRIDES     = [4, 8, 16]
ANCHORS_ORIGINALS = [
    [[5, 7],    [8, 15],   [17, 12]],
    [[15, 31],  [31, 23],  [30, 60]],
    [[58, 45],  [78, 99],  [187, 163]]
]
GRID_SIZES = [(IMG_SIZE // s, IMG_SIZE // s) for s in STRIDES]
ANCHORS    = [torch.tensor(a, dtype=torch.float32) for a in ANCHORS_ORIGINALS]

DRIVE_DIR  = "/content/drive/MyDrive/fire/model-1"
MODEL_PATH = os.path.join(DRIVE_DIR, "model-1.pth")


EVAL_SET_PATH = os.path.join(DRIVE_DIR, "evaluation_set")
IMG_DIR       = os.path.join(EVAL_SET_PATH, "images")
MASK_DIR      = os.path.join(EVAL_SET_PATH, "masks")


if not (os.path.isdir(IMG_DIR) and len(os.listdir(IMG_DIR)) > 0):
    raise FileNotFoundError(
        f"No evaluation images found in {IMG_DIR}. "
        f"Run the training script first so it saves the evaluation set."
    )
if not os.path.isdir(MASK_DIR):
    raise FileNotFoundError(f"Mask folder not found: {MASK_DIR}")


def get_id(filename):
    filename = filename.split('.')[0]
    return filename.split('_')[1]

def burn_grade(mask_name):
    mask_name = mask_name.split('.')[0]
    return int(mask_name.split('_')[3][1])

def get_box(mask):
    if mask.max() == 0:
        return None
    ys, xs = np.where(mask != 0)
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)

def to_yolo(xmin, ymin, xmax, ymax):
    w  = xmax - xmin
    h  = ymax - ymin
    xc = (xmin + xmax) / 2.0 / 320
    yc = (ymin + ymax) / 2.0 / 320
    wn = min(w * 1.1 / 320, 1.0)
    hn = min(h * 1.1 / 320, 1.0)
    return xc, yc, wn, hn


class Conv(nn.Module):
    def __init__(self, in_ch, out_ch, k, s, p):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, p)
        self.bn   = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        return F.silu(self.bn(self.conv(x)))

class Bottleneck(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.cv1 = Conv(channels, channels, 1, 1, 0)
        self.cv2 = Conv(channels, channels, 3, 1, 1)
    def forward(self, x):
        return x + self.cv2(self.cv1(x))

class C3(nn.Module):
    def __init__(self, n, in_ch, out_ch=None):
        super().__init__()
        out_ch      = out_ch or in_ch
        hidden      = out_ch // 2
        self.cv1    = Conv(in_ch, hidden, 1, 1, 0)
        self.cv2    = Conv(in_ch, hidden, 1, 1, 0)
        self.cv3    = Conv(out_ch, out_ch, 1, 1, 0)
        self.btlnck = Bottleneck(hidden)
        self.n      = n
    def forward(self, x):
        y1 = self.btlnck(self.cv1(x))
        y2 = self.cv2(x)
        for _ in range(self.n):
            y1 = self.btlnck(y1)
        return self.cv3(torch.cat([y1, y2], dim=1))

class SPPF(nn.Module):
    def __init__(self, in_ch, pool_size=5):
        super().__init__()
        hidden    = in_ch // 4
        self.cv1  = Conv(in_ch, hidden, 3, 1, 1)
        self.cv2  = Conv(hidden * 4, in_ch, 3, 1, 1)
        self.pool = nn.MaxPool2d(pool_size, stride=1, padding=pool_size // 2)
    def forward(self, x):
        x  = self.cv1(x)
        y1 = self.pool(x)
        y2 = self.pool(y1)
        y3 = self.pool(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], dim=1))

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.cv1  = Conv(3, 64, 6, 2, 2)
        self.cv2  = Conv(64, 128, 3, 1, 1)
        self.c3_1 = C3(1, 128, 128)
        self.cv3  = Conv(128, 256, 3, 2, 1)
        self.c3_2 = C3(3, 256, 256)
        self.cv4  = Conv(256, 512, 3, 2, 1)
        self.c3_3 = C3(3, 512, 512)
        self.cv5  = Conv(512, 1024, 3, 2, 1)
        self.c3_4 = C3(3, 1024, 1024)
        self.sppf = SPPF(1024)
    def forward(self, x):
        x  = self.c3_1(self.cv2(self.cv1(x)))
        p3 = self.c3_2(self.cv3(x))
        p4 = self.c3_3(self.cv4(p3))
        p5 = self.sppf(self.c3_4(self.cv5(p4)))
        return p3, p4, p5

class PANet(nn.Module):
    def __init__(self, num_classes=4, anchors_per_scale=3):
        super().__init__()
        out_ch       = anchors_per_scale * (num_classes + 5)
        c3, c4, c5   = 256, 512, 1024
        self.lat_p5  = Conv(c5, c4, 3, 1, 1)
        self.up      = nn.Upsample(None, scale_factor=2, mode="nearest")
        self.fuse_p4 = C3(1, c4 * 2, c4)
        self.lat_p4  = Conv(c4, c3, 3, 1, 1)
        self.up2     = nn.Upsample(None, scale_factor=2, mode="nearest")
        self.fuse_p3 = C3(1, c3 * 2, c3)
        self.down_p3 = Conv(c3, c3, 3, 2, 1)
        self.fuse_n4 = C3(1, c3 * 2, c4)
        self.down_n4 = Conv(c4, c4, 3, 2, 1)
        self.fuse_n5 = C3(1, c4 * 2, c5)
        self.head_s  = nn.Conv2d(c3, out_ch, 1, 1, 0)
        self.head_m  = nn.Conv2d(c4, out_ch, 1, 1, 0)
        self.head_l  = nn.Conv2d(c5, out_ch, 1, 1, 0)
    def forward(self, p3, p4, p5):
        p5r = self.lat_p5(p5)
        p4f = self.fuse_p4(torch.cat([self.up(p5r), p4], dim=1))
        p4r = self.lat_p4(p4f)
        n3  = self.fuse_p3(torch.cat([self.up(p4r), p3], dim=1))
        n4  = self.fuse_n4(torch.cat([self.down_p3(n3), p4r], dim=1))
        n5  = self.fuse_n5(torch.cat([self.down_n4(n4), p5r], dim=1))
        return self.head_s(n3), self.head_m(n4), self.head_l(n5)

class DetectionHead(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.nc = num_classes;  self.no = num_classes + 5;  self.na = 3
        self.strides = STRIDES;  self.anchors = ANCHORS
    def forward(self, feature_maps):
        p3, p4, p5 = feature_maps
        return torch.cat([self._decode(p3, 0), self._decode(p4, 1), self._decode(p5, 2)], dim=1)
    def _decode(self, raw, scale_idx):
        B, _, H, W = raw.shape
        stride  = self.strides[scale_idx]
        anchors = self.anchors[scale_idx].to(raw.device).float()
        raw     = raw.view(B, self.na, self.no, H, W).permute(0, 1, 3, 4, 2).contiguous()
        pred    = torch.sigmoid(raw)
        gy, gx  = torch.meshgrid(torch.arange(H, device=raw.device),
                                  torch.arange(W, device=raw.device), indexing='ij')
        bx  = (2.0 * pred[..., 0] - 0.5 + gx.view(1,1,H,W).float()) * stride
        by  = (2.0 * pred[..., 1] - 0.5 + gy.view(1,1,H,W).float()) * stride
        bw  = (2.0 * pred[..., 2]) ** 2 * anchors[:, 0].view(1, self.na, 1, 1)
        bh  = (2.0 * pred[..., 3]) ** 2 * anchors[:, 1].view(1, self.na, 1, 1)
        out = torch.cat([torch.stack([bx, by, bw, bh], dim=-1), pred[..., 4:]], dim=-1)
        return out.view(B, self.na * H * W, self.no)

class YOLOv5(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = Backbone()
        self.neck     = PANet(num_classes=num_classes)
        self.head     = DetectionHead(num_classes=num_classes)
    def forward(self, x):
        return self.head(self.neck(*self.backbone(x)))


def box_iou(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(0)
    lt    = torch.max(boxes1[:, None, :2], boxes2[None, :, :2])
    rb    = torch.min(boxes1[:, None, 2:], boxes2[None, :, 2:])
    inter = (rb - lt).clamp(0).prod(dim=2)
    return inter / (area1[:, None] + area2[None, :] - inter + 1e-7)

def non_max_suppression(pred, conf_thresh=0.25, iou_thresh=0.45, nc=4):
    pred = pred[pred[:, 4] > conf_thresh]
    if pred.shape[0] == 0:
        return torch.zeros((0, 6), device=pred.device)
    cls_conf, cls_id = pred[:, 5:].max(dim=1)
    score = pred[:, 4] * cls_conf
    mask  = score > conf_thresh
    pred, score, cls_id = pred[mask], score[mask], cls_id[mask]
    if pred.shape[0] == 0:
        return torch.zeros((0, 6), device=pred.device)
    cx, cy, w, h = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]
    boxes = torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=1)
    order, final = score.argsort(descending=True), []
    while order.numel() > 0:
        i = order[0]
        final.append(torch.cat([boxes[i], score[i:i+1], cls_id[i].float().view(1)]))
        if order.numel() == 1:
            break
        order = order[1:][box_iou(boxes[i:i+1], boxes[order[1:]])[0] <= iou_thresh]
    return torch.stack(final) if final else torch.zeros((0, 6), device=pred.device)


class PredictDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir   = img_dir
        self.mask_dir  = mask_dir
        self.img_dict  = {get_id(f): f for f in sorted(os.listdir(img_dir))}
        self.mask_dict = {}
        for f in sorted(os.listdir(mask_dir)):
            self.mask_dict.setdefault(get_id(f), []).append(f)
        self.ids = list(self.img_dict.keys())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        img_id  = self.ids[index]
        img     = cv2.imread(os.path.join(self.img_dir, self.img_dict[img_id]))
        img     = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img     = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        img_t   = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        masks   = []
        for mask_name in self.mask_dict.get(img_id, []):
            mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
            bbox = get_box(mask)
            if bbox is None:
                continue
            x, y, w, h = to_yolo(*bbox)
            masks.append([x, y, w, h, burn_grade(mask_name)])
        gt = (torch.tensor(masks, dtype=torch.float32)
              if masks else torch.zeros((0, 5), dtype=torch.float32))
        return img_t, gt, self.img_dict[img_id]


import ipywidgets as widgets
from IPython.display import display, clear_output

COLORS    = [(255, 60, 60), (60, 180, 255), (60, 220, 60), (0, 0, 0)]
COLOR_HEX = ["#ff3c3c", "#3cb4ff", "#3cdc3c", "#222222"]


CONF_THRESH = 0.25
IOU_THRESH  = 0.45
NUM_CLASSES = 4
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLOv5(num_classes=NUM_CLASSES)
ckpt  = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.to(device).eval()

epoch_str = (f"Epoch {ckpt['epoch'] + 1}  ·  best mAP@0.5 {ckpt.get('best_mAP', float('nan')):.4f}"
             f"  ·  eval loss {ckpt.get('best_eval_loss', float('nan')):.4f}")

dataset = PredictDataset(IMG_DIR, MASK_DIR)
n       = len(dataset)
queue   = list(range(n))
random.shuffle(queue)
state   = {"queue": queue, "pos": 0}

print(f"Loaded model-1  |  {n} eval samples")

legend_items = "".join(
    f'<span style="display:inline-flex;align-items:center;gap:5px;margin-right:18px">'
    f'<span style="width:12px;height:12px;border-radius:2px;background:{COLOR_HEX[i]};'
    f'border:1px solid rgba(255,255,255,0.25);display:inline-block"></span>'
    f'<span style="color:#ccc;font-size:12px;font-family:sans-serif">{CLASS_NAMES[i]}</span>'
    f'</span>'
    for i in range(NUM_CLASSES)
)

w_header = widgets.HTML(f"""
<div style="background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%);
            padding:16px 22px;border-radius:8px 8px 0 0;
            border-bottom:2px solid #e94560;
            display:flex;align-items:center;gap:14px">
  <span style="font-size:30px">🔥</span>
  <div style="font-family:sans-serif">
    <div style="color:#ffffff;font-size:18px;font-weight:700;letter-spacing:0.3px">
      Burn Grade Detection Viewer  —  model-1
    </div>
    <div style="color:#999;font-size:11px;margin-top:4px">
      {epoch_str} &nbsp;·&nbsp;
      conf&nbsp;{CONF_THRESH} &nbsp;·&nbsp; iou&nbsp;{IOU_THRESH}
      &nbsp;·&nbsp; {n} images
    </div>
  </div>
</div>""")

w_legend = widgets.HTML(f"""
<div style="background:#16213e;padding:9px 22px;border-bottom:1px solid #2d2d5e;
            display:flex;align-items:center;font-family:sans-serif">
  <span style="color:#666;font-size:10px;text-transform:uppercase;
               letter-spacing:1.2px;margin-right:16px">Legend</span>
  {legend_items}
  <span style="margin-left:auto;color:#666;font-size:11px">
    ─── prediction &nbsp;|&nbsp; - - - ground truth
  </span>
</div>""")

w_btn = widgets.Button(
    description="▶  Next Image",
    layout=widgets.Layout(width="148px", height="38px"),
)
w_btn.style.button_color = "#e94560"
w_btn.style.font_weight  = "bold"

w_imgname = widgets.HTML(
    '<div style="color:#888;font-size:12px;font-family:sans-serif;font-style:italic">'
    'Press the button to view the first image…</div>'
)

w_controls = widgets.HBox(
    [w_btn, w_imgname],
    layout=widgets.Layout(padding="12px 22px", align_items="center")
)

w_out = widgets.Output()

panel = widgets.VBox(
    [w_header, w_legend, w_controls, w_out],
    layout=widgets.Layout(border="1px solid #2d2d5e", width="100%")
)


def draw_boxes(ax, disp, boxes, title):
    ax.imshow(disp, extent=[0, IMG_SIZE, IMG_SIZE, 0])
    ax.set_xlim(0, IMG_SIZE);  ax.set_ylim(IMG_SIZE, 0)
    ax.set_title(title, fontsize=11, color="#ffffff", pad=8)
    ax.axis("off");  ax.set_facecolor("#0d0d1a")
    if not boxes:
        ax.text(IMG_SIZE / 2, IMG_SIZE / 2, "No detections",
                ha="center", va="center", fontsize=13, color="white",
                bbox=dict(boxstyle="round", facecolor="#e94560", alpha=0.85))
        return
    for (x1, y1, x2, y2, cls_id, linestyle, label) in boxes:
        color = tuple(c / 255.0 for c in COLORS[cls_id])
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none", linestyle=linestyle))
        ax.text((x1 + x2) / 2, y1 - 4, label,
                color="white", fontsize=8, fontweight="bold",
                ha="center", va="bottom", clip_on=True,
                bbox=dict(facecolor=color, alpha=0.85, pad=2,
                          edgecolor="none", boxstyle="round,pad=0.3"))


def on_click(_):
    if state["pos"] >= len(state["queue"]):
        state["queue"] = list(range(n))
        random.shuffle(state["queue"])
        state["pos"] = 0

    idx           = state["queue"][state["pos"]]
    state["pos"] += 1

    img_tensor, gt, img_name = dataset[idx]
    disp = np.clip(img_tensor.permute(1, 2, 0).cpu().numpy(), 0, 1)

    w_imgname.value = (
        f'<div style="color:#7ec8e3;font-size:12px;font-family:monospace">'
        f'📷 {img_name}</div>'
    )

    with torch.no_grad():
        preds = model(img_tensor.unsqueeze(0).to(device))
    dets = non_max_suppression(
        preds[0].float(), conf_thresh=CONF_THRESH, iou_thresh=IOU_THRESH, nc=NUM_CLASSES
    ).cpu().numpy()

    gt_np      = gt.cpu().numpy()
    pred_boxes = [
        (x1, y1, x2, y2, int(c), "solid", f"{CLASS_NAMES[int(c)]} {s:.2f}")
        for x1, y1, x2, y2, s, c in dets
    ]
    gt_draw = [
        (cx*IMG_SIZE - ww*IMG_SIZE/2, cy*IMG_SIZE - hh*IMG_SIZE/2,
         cx*IMG_SIZE + ww*IMG_SIZE/2, cy*IMG_SIZE + hh*IMG_SIZE/2,
         int(c), "--", CLASS_NAMES[int(c)])
        for cx, cy, ww, hh, c in gt_np
    ]

    with w_out:
        clear_output(wait=True)
        fig, (ax_pred, ax_gt) = plt.subplots(
            1, 2, figsize=(16, 8), facecolor="#0d0d1a",
            gridspec_kw={"wspace": 0.06}
        )
        draw_boxes(ax_pred, disp, pred_boxes, f"Prediction  ({len(pred_boxes)} box(es))")
        draw_boxes(ax_gt,   disp, gt_draw,    f"Ground Truth  ({len(gt_draw)} box(es))")
        fig.text(0.5, 0.01, img_name, ha="center", fontsize=9,
                 color="#555", fontfamily="monospace")
        plt.show()


w_btn.on_click(on_click)
display(panel)

Mounted at /content/drive
eval images: 23
eval masks:  48
Loaded model-1  |  23 eval samples
